# 🌸 Genshin Impact Hoyolab Wiki AI Agent

This notebook lets you:
1. **Scrape** Hoyolab Genshin Impact wiki pages and save them as local `.txt` files
2. **Ask Deepseek** questions — it reads the saved files to answer you

---
## Setup

Install required libraries:

In [ ]:
# !pip install google-generativeai requests beautifulsoup4 playwright nest_asyncio
# !pip install playwright
# !playwright install chromium

## 1. Configuration

Set your **Gemini API Key** (get one free at https://aistudio.google.com/app/apikey):

In [1]:
import os

# --- CONFIGURE THESE ---
DEEPSEEK_API_KEY = "deepseek-API-key"   # <-- Paste your Deepseek API key
WIKI_DATA_DIR  = "./wiki_data"                 # Folder where scraped .txt files are saved
# -----------------------

os.makedirs(WIKI_DATA_DIR, exist_ok=True)
print(f"✅ Wiki data will be saved to: {os.path.abspath(WIKI_DATA_DIR)}")

✅ Wiki data will be saved to: C:\Users\Anisa\OneDrive\Documents\Genshin AI Agent\wiki_data


In [2]:
import asyncio
import sys

# Force the Proactor event loop for Windows; fix Playwright asyncio issue on Windows
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


## 2. Wiki Scraper

Scrapes a Hoyolab Genshin Impact wiki page and saves the text locally.

**How to find wiki URLs:**
- Go to https://wiki.hoyolab.com/pc/genshin/home
- Navigate to any character, weapon, or game mechanic page
- Copy that URL and paste it below

In [ ]:
from playwright.sync_api import sync_playwright
from pathlib import Path
import re
import time

def sanitize_filename(name: str) -> str:
    """Turn a page title or URL slug into a safe filename."""
    name = re.sub(r'[^\w\s-]', '', name).strip()
    name = re.sub(r'[\s-]+', '_', name)
    return name[:80]


def _scrape_wiki_page(url: str, label: str = None) -> str:
    """
    Scrapes a Hoyolab wiki page using a headless browser.
    Uses the sync Playwright API to avoid asyncio issues on Windows + Python 3.13.
    """
    print(f"🌐 Opening: {url}")

    with sync_playwright() as p:
        browser = p.chromium.launch(channel="chrome",headless=True)
        page = browser.new_page(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/120.0.0.0 Safari/537.36"
        )

        page.goto(url, wait_until="domcontentloaded", timeout=30000)

        # Wait longer for Hoyolab's JS to fully render
        page.wait_for_timeout(6000)

        # Wait specifically for Hoyolab wiki content
        try:
            page.wait_for_selector(
                ".wiki-entry-content, .entry-detail, .wiki-detail, "
                ".obc-tmpl, [class*='wiki'], [class*='entry'], [class*='detail']",
                timeout=20000
            )
            print("  ✅ Wiki content detected.")
        except Exception:
            print("  ⚠️  Could not detect wiki content — scraping whatever is loaded.")

        # Scroll down to trigger any lazy-loaded content
        page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        page.wait_for_timeout(2000)  # Wait for lazy content to load after scroll


        # Get page title
        title = page.title()
        title = title.replace(" - HoYoLAB", "").replace(" | HoYoLAB", "").strip()

        # Extract visible text, removing clutter
        content = page.evaluate("""
            () => {
                const remove = ['nav', 'footer', 'script', 'style',
                                'header', 'aside', '[class*="ad"]',
                                '[class*="banner"]', '[class*="sidebar"]'];
                remove.forEach(sel => {
                    document.querySelectorAll(sel).forEach(el => el.remove());
                });
                return document.body.innerText;
            }
        """)

        browser.close()

    # Clean up whitespace
    lines = [line.strip() for line in content.splitlines()]
    lines = [l for l in lines if l]
    clean_text = "\n".join(lines)

    # Determine filename
    file_label = label if label else sanitize_filename(title if title else url.split("/")[-1])
    file_path = Path(WIKI_DATA_DIR) / f"{file_label}.txt"

    # Save with metadata header
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"SOURCE URL: {url}\n")
        f.write(f"PAGE TITLE: {title}\n")
        f.write(f"{'='*60}\n\n")
        f.write(clean_text)

    print(f"✅ Saved: {file_path}  ({len(clean_text):,} characters)")
    return str(file_path)


import threading

def scrape_wiki_page(url: str, label: str = None) -> str:
    """
    Runs the sync Playwright scraper in a separate thread to avoid
    Jupyter's background asyncio loop blocking it.
    """
    result = [None]
    error  = [None]

    def run():
        try:
            result[0] = _scrape_wiki_page(url, label)
        except Exception as e:
            error[0] = e

    thread = threading.Thread(target=run)
    thread.start()
    thread.join()

    if error[0]:
        raise error[0]
    return result[0]


print("✅ Scraper ready.")


### 2a. Scrape Individual Pages

Paste any Hoyolab wiki URL below and run the cell:

In [ ]:
# --- Paste a Hoyolab wiki URL here ---
scrape_wiki_page(
    url   = "https://wiki.hoyolab.com/pc/genshin/entry/34",  # Example: Hu Tao
    label = "hu_tao"   # Optional: custom filename (no spaces)
)

### 2b. Batch-Scrape Multiple Pages

Add as many `(url, label)` pairs as you like:

In [ ]:
def find_max_entry_id(max_probe=20000, step=1000):
    """
    Probes by checking specific IDs rather than assuming
    consecutive entries — handles gaps in Hoyolab's entry IDs.
    """
    print("🔍 Probing for max entry ID...")
    
    last_valid = 1
    
    for probe_id in range(step, max_probe + step, step):
        url = f"https://sg-wiki-api.hoyolab.com/hoyowiki/genshin/wapi/entry_page?entry_page_id={probe_id}"
        try:
            resp = requests.get(url, headers={"x-rpc-language": "en-us"}, timeout=10)
            data = resp.json()
            
            if data.get("retcode") == 0 and data.get("data"):
                print(f"  ✅ Entry {probe_id} exists")
                last_valid = probe_id
            else:
                print(f"  ❌ Entry {probe_id} not found (retcode: {data.get('retcode')})")
        except Exception as e:
            print(f"  ⚠️ Error checking {probe_id}: {e}")
        
        time.sleep(0.2)  # Small delay to avoid rate limiting
    
    print(f"\n✅ Last confirmed valid entry: {last_valid}")
    print(f"   (There may be valid entries beyond this — increase max_probe if needed)")
    return last_valid

max_id = find_max_entry_id(max_probe=20000, step=1000)

In [ ]:
# Set your base URL and the range of IDs you want
base_url = "https://wiki.hoyolab.com/pc/genshin/entry/"
start_id = 2782
end_id   = 10412  # Scrape entries 1 through 1000

for entry_id in range(start_id, end_id + 1):
    url = f"{base_url}{entry_id}"
    try:
        scrape_wiki_page(url, label=f"entry_{entry_id}")
    except Exception as e:
        print(f"❌ Failed entry {entry_id}: {e}")
    time.sleep(1)  # Small delay to avoid hammering the server

### 2c. View All Saved Files

In [ ]:
saved_files = list(Path(WIKI_DATA_DIR).glob("*.txt"))

if not saved_files:
    print("⚠️  No files saved yet. Run the scraper cells above first.")
else:
    print(f"📂 {len(saved_files)} file(s) in {WIKI_DATA_DIR}:\n")
    for f in sorted(saved_files):
        size_kb = f.stat().st_size / 1024
        print(f"  • {f.name:<40} {size_kb:>6.1f} KB")

---
## 3. Deepseek Agent

The agent loads the saved `.txt` files and uses Deepseek to answer questions.

In [3]:
from openai import OpenAI
from pathlib import Path

# Configure DeepSeek client (OpenAI-compatible API)
client = OpenAI(
    api_key  = DEEPSEEK_API_KEY,
    base_url = "https://api.deepseek.com",
)

DEEPSEEK_MODEL = "deepseek-v4-flash"   # DeepSeek-V3 — fast, large context
                                    # Use "deepseek-reasoner" for DeepSeek-R1 (slower, better reasoning)

SYSTEM_PROMPT = (
    "You are a knowledgeable Genshin Impact assistant. "
    "You will be provided with text extracted from the Hoyolab Genshin Impact wiki. "
    "Use ONLY the provided wiki content to answer the user's question. "
    "If the information isn't in the provided content, say so clearly. "
    "Be concise, accurate, and helpful."
)



def load_wiki_files(filenames: list = None) -> str:
    """
    Load wiki text files into a single context string.
    
    - filenames=None  → loads ALL .txt files in WIKI_DATA_DIR
    - filenames=['hu_tao', 'amber']  → loads only those files
    """
    data_path = Path(WIKI_DATA_DIR)
    
    if filenames:
        files = [data_path / (f if f.endswith(".txt") else f + ".txt") for f in filenames]
    else:
        files = sorted(data_path.glob("*.txt"))
    
    if not files:
        return ""
    
    parts = []
    for fp in files:
        if fp.exists():
            text = fp.read_text(encoding="utf-8")
            parts.append(f"\n\n{'='*60}\nFILE: {fp.name}\n{'='*60}\n{text}")
        else:
            print(f"  ⚠️  File not found: {fp}")
    
    return "\n".join(parts)


def ask(question: str, files: list = None, verbose: bool = False) -> str:
    """
    Ask Deepseek a question using the loaded wiki files as context.
    
    Args:
        question : Your question about Genshin Impact
        files    : List of file labels to load (None = load all)
        verbose  : If True, show which files were loaded
    
    Returns:
        Deepseek's answer as a string
    """
    wiki_context = load_wiki_files(files)
    
    if not wiki_context:
        return ("⚠️  No wiki files found. "
                "Please run the scraper cells above to save some wiki pages first.")
    
    loaded = files if files else [f.name for f in Path(WIKI_DATA_DIR).glob("*.txt")]
    if verbose:
        print(f"📚 Loaded {len(loaded)} file(s): {', '.join(str(l) for l in loaded)}")
        print(f"📝 Context size: {len(wiki_context):,} characters\n")
    
    prompt = (
        f"Here is content from the Genshin Impact Hoyolab wiki:\n"
        f"{wiki_context}\n\n"
        f"Based on the wiki content above, please answer this question:\n"
        f"{question}"
    )
    
    response = client.chat.completions.create(
        model    = DEEPSEEK_MODEL,
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature = 0.0,   # Deterministic answers for factual wiki queries
        stream      = False,
    )

    return response.choices[0].message.content

print("✅ Deepseek agent ready. Use ask('your question') to query the wiki.")

✅ Deepseek agent ready. Use ask('your question') to query the wiki.


In [4]:
import tiktoken

# ── DeepSeek-V4-Flash context limits ──────────────────────────────────────────
CONTEXT_WINDOW    = 1_000_000  # Max tokens the model can receive as input
MAX_OUTPUT_TOKENS =    32_000  # Max tokens the model can generate in its response
# Note: V4-Flash thinking mode uses tokens for its internal <think>...</think>
# reasoning chain — this counts against the output limit, not the input limit.

# ── Tokenise your wiki data ────────────────────────────────────────────────────
enc = tiktoken.get_encoding("cl100k_base")

wiki_context   = load_wiki_files()
prompt_wrapper = (
    "Here is content from the Genshin Impact Hoyolab wiki:\n"
    "\n\nBased on the wiki content above, please answer this question:\n"
    "<question>"
)

system_tokens  = len(enc.encode(SYSTEM_PROMPT))
context_tokens = len(enc.encode(wiki_context))
wrapper_tokens = len(enc.encode(prompt_wrapper))
total_tokens   = system_tokens + context_tokens + wrapper_tokens

# ── Per-file breakdown ─────────────────────────────────────────────────────────
files = sorted(Path(WIKI_DATA_DIR).glob("*.txt"))
print(f"{'FILE':<45} {'TOKENS':>8}  {'CHARS':>10}")
print("─" * 68)
for fp in files:
    text   = fp.read_text(encoding="utf-8")
    tokens = len(enc.encode(text))
    print(f"  {fp.name:<43} {tokens:>8,}  {len(text):>10,}")

# ── Summary ────────────────────────────────────────────────────────────────────
print()
print(f"{'─'*68}")
print(f"  {'System prompt':<43} {system_tokens:>8,}")
print(f"  {'Prompt wrapper':<43} {wrapper_tokens:>8,}")
print(f"  {'Wiki context (all files)':<43} {context_tokens:>8,}")
print(f"  {'TOTAL INPUT TOKENS':<43} {total_tokens:>8,}")
print(f"{'─'*68}")
print(f"  {'Context window  (deepseek-v4-flash)':<43} {CONTEXT_WINDOW:>8,}")
print(f"  {'Max output tokens':<43} {MAX_OUTPUT_TOKENS:>8,}")
print(f"  {'Tokens remaining for answer headroom':<43} {CONTEXT_WINDOW - total_tokens:>8,}")
print()

if total_tokens > CONTEXT_WINDOW:
    over = total_tokens - CONTEXT_WINDOW
    print(f"🚨 OVER LIMIT by {over:,} tokens — load specific files instead of all:")
    print("   ask('question', files=['hu_tao'])  ← only loads that one file")
elif total_tokens > CONTEXT_WINDOW * 0.85:
    pct = total_tokens / CONTEXT_WINDOW * 100
    print(f"⚠️  Using {pct:.0f}% of the context window — getting close to the limit.")
    print("   Consider using  ask('question', files=['file1', 'file2'])  to narrow scope.")
else:
    pct = total_tokens / CONTEXT_WINDOW * 100
    print(f"✅ Using {pct:.1f}% of the context window — well within limits.")

FILE                                            TOKENS       CHARS
────────────────────────────────────────────────────────────────────
  entry_1.txt                                    5,025      15,468
  entry_10.txt                                   8,750      31,524
  entry_100.txt                                     59         262
  entry_10000.txt                                   57         227
  entry_10001.txt                                   50         206
  entry_10002.txt                                   62         240
  entry_10003.txt                                   47         197
  entry_10004.txt                                   42         181
  entry_10005.txt                                   42         181
  entry_10006.txt                                   41         180
  entry_10007.txt                                   42         180
  entry_10008.txt                                   53         219
  entry_10009.txt                                   54      

---
## 4. Map-Reduce Q&A  *(handles datasets larger than the context window)*

When your total wiki data exceeds the model's context limit, `ask_map_reduce()` solves it by:

1. **Map** — splits all wiki text into smaller chunks and sends each to DeepSeek individually
2. **Reduce** — synthesises all partial answers into one final, coherent answer

> 💡 **Tip:** Use `files=[...]` to narrow scope (e.g. `files=['hu_tao', 'neuvillette']`) — fewer chunks = faster + cheaper.

In [9]:
# ──────────────────────────────────────────────────────────────────────────
# Map-Reduce Q&A  — works even when total data exceeds the context window
# ──────────────────────────────────────────────────────────────────────────

REDUCE_SYSTEM_PROMPT = (
    "You are a Genshin Impact expert synthesising research notes. "
    "Combine the partial answers below into one accurate, non-redundant response. "
    "Resolve contradictions by preferring the most detailed answer. "
    "Remove repetition and organise the information clearly."
)


def ask_map_reduce(
    question: str,
    files: list = None,
    chunk_tokens: int = 60_000,   # Target tokens per chunk (1 token ≈ 4 chars)
    max_partial_tokens: int = 512,  # Max tokens per partial answer
    max_final_tokens: int = 1024,   # Max tokens for the synthesised answer
    verbose: bool = True,
) -> str:
    """
    Map-Reduce RAG: splits wiki context into chunks that fit the model,
    queries each chunk individually, then synthesises all partial answers
    into one final answer.

    Args:
        question          : Your Genshin Impact question.
        files             : File labels to load (None = all files).
        chunk_tokens      : Approx token budget per map call  (default 60 k).
        max_partial_tokens: Max tokens in each partial answer (default 512).
        max_final_tokens  : Max tokens in the synthesised answer (default 1024).
        verbose           : Print progress (default True).

    Returns:
        Synthesised answer string.
    """
    # ── Load wiki files ──────────────────────────────────────────────────────
    wiki_context = load_wiki_files(files)
    if not wiki_context:
        return ("⚠️  No wiki files found. "
                "Run the scraper cells first, or check your WIKI_DATA_DIR.")

    # ── Split into text chunks (1 token ≈ 4 chars) ──────────────────────────
    chars_per_chunk = chunk_tokens * 4
    chunks = [
        wiki_context[i : i + chars_per_chunk]
        for i in range(0, len(wiki_context), chars_per_chunk)
    ]

    if verbose:
        total_tokens_est = len(wiki_context) // 4
        print(f"📂 Context : {len(wiki_context):>12,} chars  (~{total_tokens_est:,} tokens)")
        print(f"📦 Chunks  : {len(chunks)} × ~{chunk_tokens:,} tokens each")
        print(f"❓ Question: {question}\n")
        print("─" * 60)

    # ── MAP: query each chunk ────────────────────────────────────────────────
    partial_answers = []
    for i, chunk in enumerate(chunks, 1):
        if verbose:
            print(f"  🔄 Chunk {i:>3}/{len(chunks)}  ({len(chunk):,} chars)", end="  →  ")

        map_prompt = (
            f"Below is part {i} of {len(chunks)} of content extracted from the "
            f"Genshin Impact Hoyolab wiki.\n\n"
            f"{chunk}\n\n"
            f"Using ONLY the wiki content shown above, answer this question as "
            f"completely as you can:\n{question}\n\n"
            f"Important: If this part contains NO information relevant to the "
            f"question, respond with exactly: NO_RELEVANT_INFO"
        )

        try:
            resp = client.chat.completions.create(
                model       = DEEPSEEK_MODEL,
                messages    = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": map_prompt},
                ],
                temperature = 0.0,
                max_tokens  = max_partial_tokens,
            )
            answer = resp.choices[0].message.content.strip()
        except Exception as e:
            answer = f"[API error on chunk {i}: {e}]"

        partial_answers.append(answer)

        if verbose:
            if answer == "NO_RELEVANT_INFO":
                print("⏭  no relevant info")
            else:
                snippet = answer[:80].replace("\n", " ")
                print(f"✅ {snippet}…")

    # ── Filter out uninformative partial answers ─────────────────────────────
    relevant = [
        a for a in partial_answers
        if a != "NO_RELEVANT_INFO"
        and not a.lower().startswith("no_relevant_info")
        and not a.lower().startswith("no relevant info")
        and len(a.strip()) > 20
    ]

    if verbose:
        print("─" * 60)
        print(f"\n📝 {len(relevant)} / {len(chunks)} chunks had relevant info.")

    if not relevant:
        return "❌ No relevant information found across any wiki chunk for that question."

    # Single relevant answer — return directly without an extra API call
    if len(relevant) == 1:
        return relevant[0]

    # ── REDUCE: synthesise all partial answers ───────────────────────────────
    if verbose:
        print(f"🔀 Synthesising {len(relevant)} partial answers…\n")

    joined = ""
    for idx, ans in enumerate(relevant, 1):
        joined += f"--- Partial answer {idx} ---\n{ans}\n\n"

    reduce_prompt = (
        f"The following are partial answers collected from different sections of the "
        f"Genshin Impact wiki to answer this question:\n"
        f"\"{ question }\"\n\n"
        f"{joined}"
        f"Synthesise these into a single, comprehensive, non-redundant final answer. "
        f"Organise the information clearly with headings or bullet points where helpful."
    )

    final_resp = client.chat.completions.create(
        model       = DEEPSEEK_MODEL,
        messages    = [
            {"role": "system", "content": REDUCE_SYSTEM_PROMPT},
            {"role": "user",   "content": reduce_prompt},
        ],
        temperature = 0.0,
        max_tokens  = max_final_tokens,
    )

    return final_resp.choices[0].message.content.strip()


print("✅ ask_map_reduce() ready.")
print("   Usage: ask_map_reduce('your question')")
print("   Tip  : ask_map_reduce('question', files=['hu_tao', 'neuvillette'])  ← faster")


✅ ask_map_reduce() ready.
   Usage: ask_map_reduce('your question')
   Tip  : ask_map_reduce('question', files=['hu_tao', 'neuvillette'])  ← faster


In [7]:
import json
import tiktoken
from pathlib import Path

# ── Config ─────────────────────────────────────────────────────────────────────
CHUNK_SIZE    = 1_500
CHUNK_OVERLAP = 50

CONTEXT_WINDOW    = 1_000_000
RESERVED_OUTPUT   =    32_000  # Headroom for answer + think chain
SAFETY_BUFFER     =     2_000  # Extra buffer for tiktoken vs API tokeniser drift
CHUNKS_FILE = Path(WIKI_DATA_DIR) / "_chunks.json"

enc = tiktoken.get_encoding("cl100k_base")

PROMPT_WRAPPER = (
    "Here is content from the Genshin Impact Hoyolab wiki:\n"
    "{context}"
    "\n\nBased on the wiki content above, please answer this question:\n"
    "{question}"
)


# ── 1. Chunker ─────────────────────────────────────────────────────────────────

def _chunk_text(text: str, source: str) -> list[dict]:
    tokens = enc.encode(text)
    chunks = []
    start  = 0
    while start < len(tokens):
        end        = min(start + CHUNK_SIZE, len(tokens))
        chunk_toks = tokens[start:end]
        chunks.append({
            "source":      source,
            "chunk_index": len(chunks),
            "token_count": len(chunk_toks),
            "text":        enc.decode(chunk_toks),
        })
        if end == len(tokens):
            break
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks


def build_chunks(force_rebuild: bool = False) -> list[dict]:
    if CHUNKS_FILE.exists() and not force_rebuild:
        all_chunks = json.loads(CHUNKS_FILE.read_text(encoding="utf-8"))
        print(f"📦 Loaded {len(all_chunks)} cached chunks from {CHUNKS_FILE.name}")
        return all_chunks

    files      = sorted(Path(WIKI_DATA_DIR).glob("*.txt"))
    all_chunks = []

    print(f"✂️  Chunking {len(files)} file(s) "
          f"(chunk={CHUNK_SIZE} tok, overlap={CHUNK_OVERLAP} tok)...\n")
    print(f"  {'FILE':<40} {'CHUNKS':>7}  {'TOKENS':>8}")
    print("  " + "─" * 58)

    for fp in files:
        text   = fp.read_text(encoding="utf-8")
        chunks = _chunk_text(text, source=fp.name)
        total  = sum(c["token_count"] for c in chunks)
        print(f"  {fp.name:<40} {len(chunks):>7}  {total:>8,}")
        all_chunks.extend(chunks)

    CHUNKS_FILE.write_text(
        json.dumps(all_chunks, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("  " + "─" * 58)
    print(f"\n  Total chunks : {len(all_chunks):,}")
    print(f"  Total tokens : {sum(c['token_count'] for c in all_chunks):,}")
    print(f"\n✅ Chunks saved → {CHUNKS_FILE}")
    return all_chunks


# ── 2. Chunk loader ────────────────────────────────────────────────────────────

def load_chunks_for_files(
    filenames:    list[str] | None = None,
    token_budget: int = None,
) -> tuple[str, dict]:
    all_chunks = json.loads(CHUNKS_FILE.read_text(encoding="utf-8"))

    if filenames:
        want = {f if f.endswith(".txt") else f + ".txt" for f in filenames}
        pool = [c for c in all_chunks if c["source"] in want]
    else:
        pool = all_chunks

    # Default budget: full window minus output reservation and safety buffer
    if token_budget is None:
        token_budget = CONTEXT_WINDOW - RESERVED_OUTPUT - SAFETY_BUFFER

    selected, used_tokens, truncated = [], 0, False
    for chunk in pool:
        if used_tokens + chunk["token_count"] > token_budget:
            truncated = True
            break
        selected.append(chunk)
        used_tokens += chunk["token_count"]

    parts, current_source = [], None
    for c in selected:
        if c["source"] != current_source:
            parts.append(f"\n{'='*60}\nFILE: {c['source']}\n{'='*60}")
            current_source = c["source"]
        parts.append(f"\n[chunk {c['chunk_index']+1}]\n{c['text']}")

    stats = {
        "total_chunks_available": len(pool),
        "chunks_loaded":          len(selected),
        "chunks_omitted":         len(pool) - len(selected),
        "tokens_used":            used_tokens,
        "token_budget":           token_budget,
        "truncated":              truncated,
    }
    return "\n".join(parts), stats


# ── 3. ask_chunked() ───────────────────────────────────────────────────────────

def ask_chunked(
    question: str,
    files:    list[str] | None = None,
    verbose:  bool = False,
) -> str:
    if not CHUNKS_FILE.exists():
        print("⚠️  No chunk cache found — running build_chunks() now...")
        build_chunks()

    # ── Step 1: measure fixed overhead tokens BEFORE loading chunks ────────────
    # This is the root cause of the original 400 error: overhead was measured
    # after filling the budget, so it was never subtracted from available space.
    system_tokens  = len(enc.encode(SYSTEM_PROMPT))
    question_tokens = len(enc.encode(question))
    wrapper_tokens = len(enc.encode(
        PROMPT_WRAPPER.replace("{context}", "").replace("{question}", question)
    ))
    overhead_tokens = system_tokens + question_tokens + wrapper_tokens

    # ── Step 2: derive safe budget for wiki chunks only ────────────────────────
    chunk_budget = CONTEXT_WINDOW - RESERVED_OUTPUT - SAFETY_BUFFER - overhead_tokens

    if verbose:
        print(f"📐 Token budget breakdown:")
        print(f"   Context window          : {CONTEXT_WINDOW:>10,}")
        print(f"   − Reserved output       : {RESERVED_OUTPUT:>10,}")
        print(f"   − Safety buffer         : {SAFETY_BUFFER:>10,}")
        print(f"   − System prompt         : {system_tokens:>10,}")
        print(f"   − Prompt wrapper        : {wrapper_tokens:>10,}")
        print(f"   − Question              : {question_tokens:>10,}")
        print(f"   ═ Available for chunks  : {chunk_budget:>10,}")
        print()

    if chunk_budget <= 0:
        return (
            f"🚨 Overhead alone ({overhead_tokens:,} tokens) exceeds the safe budget. "
            "Shorten your system prompt or question."
        )

    # ── Step 3: load chunks within the safe budget ─────────────────────────────
    context_str, stats = load_chunks_for_files(filenames=files, token_budget=chunk_budget)

    if not context_str.strip():
        return "⚠️  No wiki content loaded. Check your WIKI_DATA_DIR."

    # ── Step 4: verify the fully assembled message before sending ──────────────
    prompt = PROMPT_WRAPPER.format(context=context_str, question=question)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    actual_tokens = sum(len(enc.encode(m["content"])) for m in messages)

    if verbose or stats["truncated"]:
        pct = stats["tokens_used"] / chunk_budget * 100
        print(f"📦 Chunks loaded  : {stats['chunks_loaded']:,} / "
              f"{stats['total_chunks_available']:,}")
        print(f"🔢 Chunk tokens   : {stats['tokens_used']:,} / "
              f"{chunk_budget:,}  ({pct:.1f}%)")
        print(f"📨 Total message  : {actual_tokens:,} / {CONTEXT_WINDOW:,} tokens")
        if stats["truncated"]:
            print(f"⚠️  {stats['chunks_omitted']} chunk(s) didn't fit. "
                  "Use files=[...] to narrow scope.")
        print()

    # Final hard stop — should never trigger given the math above
    if actual_tokens > CONTEXT_WINDOW - RESERVED_OUTPUT:
        return (
            f"🚨 Assembled message ({actual_tokens:,} tokens) still exceeds safe limit. "
            f"Try: ask_chunked('question', files=['neuvillette'])"
        )

    # ── Step 5: send ───────────────────────────────────────────────────────────
    response = client.chat.completions.create(
        model       = DEEPSEEK_MODEL,
        messages    = messages,
        temperature = 0.0,
        stream      = False,
    )
    return response.choices[0].message.content


# ── 4. Run ─────────────────────────────────────────────────────────────────────
chunks = build_chunks()

context_str, stats = load_chunks_for_files()
pct = stats["tokens_used"] / stats["token_budget"] * 100
print(f"\n📊 Full dataset: {stats['tokens_used']:,} / {stats['token_budget']:,} "
      f"chunk-budget tokens  ({pct:.1f}%)")

📦 Loaded 10242 cached chunks from _chunks.json

📊 Full dataset: 965,497 / 966,000 chunk-budget tokens  (99.9%)


---
## 4. Ask Questions!

### Simple query (searches all saved files)

In [10]:
# ── Example 1: All files (slow — processes every chunk across the full dataset) ──
answer = ask_map_reduce(
    question = "What is the best artifact set for Neuvillette's DPS build?",
    verbose  = True,
)
print("\n🌸 Final Answer:\n")
print(answer)


📂 Context :   16,160,072 chars  (~4,040,018 tokens)
📦 Chunks  : 68 × ~60,000 tokens each
❓ Question: What is the best artifact set for Neuvillette's DPS build?

────────────────────────────────────────────────────────────
  🔄 Chunk   1/68  (240,000 chars)  →  ⏭  no relevant info
  🔄 Chunk   2/68  (240,000 chars)  →  ⏭  no relevant info
  🔄 Chunk   3/68  (240,000 chars)  →  ⏭  no relevant info
  🔄 Chunk   4/68  (240,000 chars)  →  ⏭  no relevant info
  🔄 Chunk   5/68  (240,000 chars)  →  ⏭  no relevant info
  🔄 Chunk   6/68  (240,000 chars)  →  ⏭  no relevant info
  🔄 Chunk   7/68  (240,000 chars)  →  ⏭  no relevant info
  🔄 Chunk   8/68  (240,000 chars)  →  ⏭  no relevant info
  🔄 Chunk   9/68  (240,000 chars)  →  ⏭  no relevant info
  🔄 Chunk  10/68  (240,000 chars)  →  ⏭  no relevant info
  🔄 Chunk  11/68  (240,000 chars)  →  ⏭  no relevant info
  🔄 Chunk  12/68  (240,000 chars)  →  ⏭  no relevant info
  🔄 Chunk  13/68  (240,000 chars)  →  ⏭  no relevant info
  🔄 Chunk  14/68  (240,0

In [8]:
# Build chunks once (re-run with force_rebuild=True if you scrape new pages)
chunks = build_chunks(force_rebuild=True)

answer = ask_chunked("What is the best artifact set for Neuvillette's dps build?", verbose=True)
print(answer)

✂️  Chunking 8672 file(s) (chunk=1500 tok, overlap=50 tok)...

  FILE                                      CHUNKS    TOKENS
  ──────────────────────────────────────────────────────────
  entry_1.txt                                    4     5,175
  entry_10.txt                                   6     9,000
  entry_100.txt                                  1        59
  entry_10000.txt                                1        57
  entry_10001.txt                                1        50
  entry_10002.txt                                1        62
  entry_10003.txt                                1        47
  entry_10004.txt                                1        42
  entry_10005.txt                                1        42
  entry_10006.txt                                1        41
  entry_10007.txt                                1        42
  entry_10008.txt                                1        53
  entry_10009.txt                                1        54
  entry_1001.txt      

### Target specific files (faster, more focused)

In [11]:
answer = ask(
    question = "What are Hu Tao's best weapons?",
    files    = ["hu_tao"],   # Only load hu_tao.txt
    verbose  = True
)
print(answer)

📚 Loaded 1 file(s): hu_tao
📝 Context size: 27,339 characters

The provided wiki content is for **Sangonomiya Kokomi**, not Hu Tao. There is no information about Hu Tao's best weapons in this document.


### Interactive chat loop

Keep asking questions until you type `quit`:

In [ ]:
print("🤖 Genshin Wiki Agent — type 'quit' to exit\n")
print(f"📂 Loaded files: {[f.name for f in Path(WIKI_DATA_DIR).glob('*.txt')]}\n")

while True:
    question = input("You: ").strip()
    if not question:
        continue
    if question.lower() in ("quit", "exit", "q"):
        print("👋 Bye!")
        break
    
    print("\n🌸 Deepseek:")
    answer = ask(question)
    print(answer)
    print("\n" + "-"*50 + "\n")

---
## 5. Utilities

### Preview a saved file

In [ ]:
def preview_file(label: str, lines: int = 50):
    fp = Path(WIKI_DATA_DIR) / f"{label}.txt"
    if not fp.exists():
        print(f"File not found: {fp}")
        return
    content = fp.read_text(encoding="utf-8").splitlines()
    print(f"\n--- {fp.name} (first {lines} lines) ---\n")
    print("\n".join(content[:lines]))
    if len(content) > lines:
        print(f"\n... ({len(content) - lines} more lines)")

preview_file("hu_tao")

### Delete a saved file

In [ ]:
def delete_file(label: str):
    fp = Path(WIKI_DATA_DIR) / f"{label}.txt"
    if fp.exists():
        fp.unlink()
        print(f"🗑️  Deleted: {fp.name}")
    else:
        print(f"File not found: {fp}")

# delete_file("hu_tao")   # Uncomment to delete

---
## Quick Reference

| Task | Code |
|------|------|
| Scrape a page | `scrape_wiki_page(url, label)` |
| List saved files | `list(Path(WIKI_DATA_DIR).glob('*.txt'))` |
| Ask using all files | `ask('your question')` |
| Ask using specific files | `ask('question', files=['hu_tao', 'amber'])` |
| Preview a file | `preview_file('hu_tao')` |
| Delete a file | `delete_file('hu_tao')` |

**Good Hoyolab wiki pages to scrape:**
- Characters: `https://wiki.hoyolab.com/pc/genshin/entry/{id}`
- Browse the wiki at: https://wiki.hoyolab.com/pc/genshin/home